# Menjalankan AI-Telebot di Google Colab
Notebook ini akan membantu Anda menjalankan `ai-telebot` beserta Ollama GGUF (sebagai Worker) langsung di Colab dengan memanfaatkan GPU gratis.

In [ ]:
# Install dependencies
!pip install -r requirements.txt

In [ ]:
# 1. Mount Google Drive dan Pindah ke Folder Project
from google.colab import drive
import os

drive.mount('/content/drive')

# GANTI INI sesuai dengan lokasi folder ai-telebot Anda di Google Drive
project_path = '/content/drive/MyDrive/ai-telebot'
os.chdir(project_path)

print(f'Current Directory: {os.getcwd()}')


In [ ]:
# 2. Install dan Jalankan Ollama di Background
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

print('Memulai server Ollama...')
process = subprocess.Popen(['ollama', 'serve'])
time.sleep(3) # Tunggu server up
print('Ollama siap!')

In [ ]:
# 3. (Opsional) Pull atau Build Model GGUF Anda
# Jika Anda ingin menggunakan qwen2.5-coder:3b langsung dari repository Ollama:
!ollama pull qwen2.5-coder:3b

# ATAU, jika Anda punya file .gguf sendiri, uncomment baris di bawah ini dan edit path-nya:
# modelfile_content = '''FROM /content/drive/MyDrive/models/qwen2.5-coder-3b.gguf'''
# with open('Modelfile', 'w') as f:
#     f.write(modelfile_content)
# !ollama create model_ku_keren -f Modelfile


In [ ]:
# 4. Bertanya Langsung ke AI-Telebot (LangGraph) secara Hardcode
import os
import sys

# Tambahkan path root agar Python mengenali folder core/
project_root = "/content/drive/MyDrive/ai-telebot"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Load environment variables dari .env (jika ada)
from dotenv import load_dotenv
load_dotenv()

from core.agent_graph import get_agent_executor
from langchain_core.messages import HumanMessage

# ========================================================
# HARDCODE PERTANYAAN ANDA DI BAWAH INI
# ========================================================
pertanyaan = "Coba buatkan file python berisi fungsi untuk menghitung Fibonacci."
session_id = "colab_session_1"
# ========================================================

print(f"👤 User: {pertanyaan}\n")
print("🤖 AI-Telebot sedang berpikir (menjalankan LangGraph Master-Worker-Reviewer)...")

# Ambil fungsi eksekutor agent graph
agent_executor = get_agent_executor(active_project="default", user_query=pertanyaan)
messages = [HumanMessage(content=pertanyaan)]

final_message = ""

# Eksekusi pipeline graph
for event in agent_executor.stream(
    {"messages": messages}, 
    config={"configurable": {"thread_id": session_id}, "recursion_limit": 100}
):
    for key, value in event.items():
        print(f"\n✅ [Node Selesai]: {key}")
        
        # Mengekstrak pesan dari node
        if "messages" in value:
            agent_msgs = value.get("messages", [])
            if not isinstance(agent_msgs, list):
                agent_msgs = [agent_msgs]
                
            for msg in agent_msgs:
                # Jangan print JSON/tool_calls mentah, simpan balasan teks murni
                if getattr(msg, "content", "") and not getattr(msg, "tool_calls", None):
                    final_message = msg.content

print("\n================ JAWABAN AKHIR AI-TELEBOT ================\n")
print(final_message)
